# 1. gpkg 파일 로드

In [8]:
import sys
print(sys.version)
print(sys.executable)

3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]
c:\ProgramData\Anaconda3\python.exe


In [9]:
import geopandas as gpd
import fiona

In [18]:
gdf = gpd.read_file("가로수_500.gpkg")
gdf.head()

,id,left,top,right,bottom,NUMPOINTS,geometry
0,69,1.077033e+06,1.746587e+06,1.077533e+06,1.746087e+06,0.0,"MULTIPOLYGON (((1077501.514 1746091.812, 10775..."
1,70,1.077033e+06,1.746087e+06,1.077533e+06,1.745587e+06,0.0,"MULTIPOLYGON (((1077057.173 1745639.546, 10771..."
2,71,1.077033e+06,1.745587e+06,1.077533e+06,1.745087e+06,0.0,"MULTIPOLYGON (((1077032.542 1745130.723, 10770..."
3,72,1.077033e+06,1.745087e+06,1.077533e+06,1.744587e+06,0.0,"MULTIPOLYGON (((1077088.942 1744663.761, 10770..."
4,73,1.077033e+06,1.744587e+06,1.077533e+06,1.744087e+06,0.0,"MULTIPOLYGON (((1077218.85 1744149.184, 107720..."


In [13]:
print(gdf.dtypes)

id              int64
left          float64
top           float64
right         float64
bottom        float64
NUMPOINTS     float64
geometry     geometry
dtype: object


In [14]:
gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 89316 entries, 0 to 89315
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   id         89316 non-null  int64   
 1   left       89316 non-null  float64 
 2   top        89316 non-null  float64 
 3   right      89316 non-null  float64 
 4   bottom     89316 non-null  float64 
 5   NUMPOINTS  89316 non-null  float64 
 6   geometry   89316 non-null  geometry
dtypes: float64(5), geometry(1), int64(1)
memory usage: 4.8 MB


In [20]:
print(gdf.iloc[0]['geometry'])

MULTIPOLYGON (((1077501.5144790818 1746091.8118976867, 1077532.5416 1746108.0900132162, 1077532.5416 1746087.4112, 1077494.674652851 1746087.4112, 1077501.5144790818 1746091.8118976867)))


# 2. 한국 좌표계 변환 TEST

In [23]:
gdf = gdf.to_crs(epsg=4326)

In [24]:
gdf['centeroid'] = gdf.geometry.centroid
gdf['lon'] = gdf.centroid.x #경도
gdf['lat'] = gdf.centroid.y #위도

C:\Users\빅데이터활용센터\AppData\Local\Temp\ipykernel_6276\3840422882.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['centeroid'] = gdf.geometry.centroid
C:\Users\빅데이터활용센터\AppData\Local\Temp\ipykernel_6276\3840422882.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['lon'] = gdf.centroid.x #경도
C:\Users\빅데이터활용센터\AppData\Local\Temp\ipykernel_6276\3840422882.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['lat'] = gdf.centroid.y #위도


In [25]:
gdf.head()

,id,left,top,right,bottom,NUMPOINTS,geometry,centeroid,lon,lat
0,69,1.077033e+06,1.746587e+06,1.077533e+06,1.746087e+06,0.0,"MULTIPOLYGON (((128.35675 35.70806, 128.3571 3...",POINT (128.35695 35.70808),128.356952,35.708080
1,70,1.077033e+06,1.746087e+06,1.077533e+06,1.745587e+06,0.0,"MULTIPOLYGON (((128.3518 35.70402, 128.3523 35...",POINT (128.35511 35.70523),128.355110,35.705226
2,71,1.077033e+06,1.745587e+06,1.077533e+06,1.745087e+06,0.0,"MULTIPOLYGON (((128.35148 35.69943, 128.35148 ...",POINT (128.35431 35.70126),128.354315,35.701256
3,72,1.077033e+06,1.745087e+06,1.077533e+06,1.744587e+06,0.0,"MULTIPOLYGON (((128.35205 35.69522, 128.35192 ...",POINT (128.35436 35.69682),128.354358,35.696825
4,73,1.077033e+06,1.744587e+06,1.077533e+06,1.744087e+06,0.0,"MULTIPOLYGON (((128.35344 35.69057, 128.3533 3...",POINT (128.35489 35.69239),128.354888,35.692391


# 3. 격자 중심점 + 개수 추출.csv

In [26]:
import geopandas as gpd
from shapely.geometry import box
import pandas as pd
from pathlib import Path

In [27]:
FILES = {
    "100m": "가로수_100.gpkg",
    "300m": "가로수_300.gpkg",
    "500m": "가로수_500.gpkg"
} 

In [30]:
MIN_LON, MAX_LON = 128.6013, 128.6206
MIN_LAT, MAX_LAT = 35.8792, 35.8933
bbox_wgs84 = box(MIN_LON, MIN_LAT, MAX_LON, MAX_LAT)

In [31]:
rows = []

for grid_size, path in FILES.items():
    gdf = gpd.read_file(path)

    # 위경도 변환
    gdf_wgs = gdf.to_crs(epsg=4326)

    # BBOX로 공간 필터
    gdf_clip = gdf_wgs[gdf_wgs.intersects(bbox_wgs84)].copy()

    # 격자 중심점(위경도) 계산
    cent = gdf_clip.geometry.centroid
    gdf_clip["lon"] = cent.x
    gdf_clip["lat"] = cent.y

    # 내보낼 최소 컬럼 구성
    out = gdf_clip[["id", "NUMPOINTS", "lon", "lat"]].copy()
    out["grid_size_m"] = int(grid_size.replace("m",""))
    rows.append(out)

C:\Users\빅데이터활용센터\AppData\Local\Temp\ipykernel_6276\200211492.py:13: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cent = gdf_clip.geometry.centroid
C:\Users\빅데이터활용센터\AppData\Local\Temp\ipykernel_6276\200211492.py:13: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cent = gdf_clip.geometry.centroid
C:\Users\빅데이터활용센터\AppData\Local\Temp\ipykernel_6276\200211492.py:13: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cent = gdf_clip.geometry.centroid


In [34]:
type(rows)

list

In [37]:
out_df = pd.concat(rows, ignore_index = True) # 기존 인덱스 무시하고, 새로은 인덱스 부여
out_df.head()

,id,NUMPOINTS,lon,lat,grid_size_m
0,102272,0.0,128.601181,35.893106,100
1,102273,0.0,128.601169,35.892205,100
2,102274,6.0,128.601156,35.891303,100
3,102275,23.0,128.601144,35.890402,100
4,102276,24.0,128.601131,35.889500,100


In [35]:
out_df.to_csv("경북대_가로수_위경도.csv", index=False)